# Gain filter sanity check

Runs `GainFilterAndBin.process_data` on a single (obsid, feed, band) and overplots the binned filtered data on the binned data, with vertical lines at scan edges.

In [ ]:
import os, sys
from pathlib import Path

# Make the project root importable when running from notebooks/
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import h5py
import matplotlib.pyplot as plt

from modules.SQLModule.SQLModule import db
from modules.ProcessLevel2.GainFilterAndBin.GainFilterAndBin import GainFilterAndBin
from modules.utils.data_handling import read_2bit

In [ ]:
# ---- inputs ----
OBSID = 30000        # <-- set obsid
FEED  = 1            # 1-indexed feed (1..19)
BAND  = 0            # 0..3
DB_PATH = 'databases/COMAP_manchester.db'

# Gain-filter config (mirror what the pipeline uses)
GF_KWARGS = dict(end_cut=50, n_freq_bin=2, gain_filter_name='GainFilterBase', gain_filter_kwargs={})

In [ ]:
# ---- look up paths ----
db.connect(DB_PATH)
from modules.SQLModule.SQLModule import COMAPData
row = db.session.query(COMAPData).filter_by(obsid=OBSID).first()
assert row is not None, f'obsid {OBSID} not in DB'
level1_path = row.level1_path
level2_path = row.level2_path
print('level1:', level1_path)
print('level2:', level2_path)
assert level1_path and os.path.exists(level1_path), 'missing level1'
assert level2_path and os.path.exists(level2_path), 'missing level2'

In [ ]:
# ---- load only what we need for this feed ----
with h5py.File(level1_path, 'r') as ds, h5py.File(level2_path, 'r') as lvl2:
    feeds = ds['spectrometer/feeds'][:]
    ifeed = int(np.where(feeds == FEED)[0][0])
    print(f'feed {FEED} -> array index {ifeed}')

    frequencies = ds['spectrometer/frequency'][:]            # (band, channel)
    scan_edges  = lvl2['level2/scan_edges'][...]

    # TOD for just this feed: (band, channel, time)
    data = ds['spectrometer/tod'][ifeed, ...].astype(np.float64)

    system_temperature = lvl2['level2/vane/system_temperature'][0, ifeed, ...]
    gain               = lvl2['level2/vane/gain'][0, ifeed, ...]
    atmos_offsets      = lvl2['level2/atmosphere/offsets'][ifeed, ...]
    atmos_tau          = lvl2['level2/atmosphere/tau'][ifeed, ...]

    # Vane-calibrate (matches GainFilterAndBin.gain_filter_and_bin)
    auto_rms = lvl2['level1_noise_stats/auto_rms'][FEED-1, ...] / gain
    data = data / gain[..., np.newaxis]

    if 'level2_noise_stats/binned_data' in lvl2:
        sigma_red = lvl2['level2_noise_stats/binned_data/sigma_red'][FEED-1, ...]
        alpha     = lvl2['level2_noise_stats/binned_data/alpha'][FEED-1, ...]
    else:
        sigma_red, alpha = None, None

    elevation = np.radians(lvl2['spectrometer/pixel_pointing/pixel_el'][ifeed, ...])

print('data shape (band, chan, time):', data.shape)
print('scan_edges:', scan_edges)

In [ ]:
# ---- run the filter on this one feed ----
gfb = GainFilterAndBin(**GF_KWARGS)
binned_data, binned_filtered_data, central_freqs, bandwidths = gfb.process_data(
    data, frequencies, system_temperature, atmos_offsets, atmos_tau,
    elevation, scan_edges, auto_rms, FEED, sigma_red, alpha,
)
print('binned shape (band, n_freq_bin, time):', binned_data.shape)

In [ ]:
# ---- plot ----
n_freq_bin = binned_data.shape[1]
fig, axes = plt.subplots(n_freq_bin, 1, figsize=(14, 3.2 * n_freq_bin), sharex=True, squeeze=False)

for ibin in range(n_freq_bin):
    ax = axes[ibin, 0]
    y_raw = binned_data[BAND, ibin, :]
    y_flt = binned_filtered_data[BAND, ibin, :]
    ax.plot(y_raw, lw=0.6, label='binned', color='C0', alpha=0.7)
    ax.plot(y_flt, lw=0.6, label='binned_filtered', color='C3', alpha=0.9)
    for s, e in scan_edges:
        ax.axvline(s, color='k', lw=0.5, ls='--', alpha=0.5)
        ax.axvline(e, color='k', lw=0.5, ls=':',  alpha=0.5)
    ax.set_ylabel(f'band {BAND}, freq bin {ibin}\n({central_freqs[BAND, ibin]:.2f} GHz)')
    ax.legend(loc='upper right', fontsize=8)

axes[-1, 0].set_xlabel('sample')
fig.suptitle(f'obsid {OBSID}  feed {FEED}  band {BAND}')
fig.tight_layout()
plt.show()

In [ ]:
# Optional: zoom into a single scan
ISCAN = 0
s, e = scan_edges[ISCAN]
fig, axes = plt.subplots(n_freq_bin, 1, figsize=(14, 3.2 * n_freq_bin), sharex=True, squeeze=False)
for ibin in range(n_freq_bin):
    ax = axes[ibin, 0]
    ax.plot(np.arange(s, e), binned_data[BAND, ibin, s:e], lw=0.6, label='binned', color='C0', alpha=0.7)
    ax.plot(np.arange(s, e), binned_filtered_data[BAND, ibin, s:e], lw=0.6, label='binned_filtered', color='C3')
    ax.axvline(s, color='k', lw=0.5, ls='--')
    ax.axvline(e, color='k', lw=0.5, ls=':')
    ax.set_ylabel(f'freq bin {ibin}')
    ax.legend(loc='upper right', fontsize=8)
axes[-1, 0].set_xlabel('sample')
fig.suptitle(f'obsid {OBSID}  feed {FEED}  band {BAND}  scan {ISCAN}')
fig.tight_layout()
plt.show()